In [19]:
import pandas as pd 
df = pd.read_csv('../Raw/January2015toNovember2025.csv',parse_dates=['EventDate'])



/var/folders/g5/pd1x146x0w776scfmklbv8x80000gn/T/ipykernel_75674/334417464.py:2: DtypeWarning: Columns (0: Primary NAICS, 1: Inspection) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('../Raw/January2015toNovember2025.csv',parse_dates=['EventDate'])


In [20]:
df = df.drop(columns=['Zip', 'Latitude', 'Longitude', 'Address1', 'Address2', 'Inspection','Employer'])


In [21]:
df = df.drop(columns=['FederalState'])


In [22]:
# Показывает информацию по всем колонкам
df.info(verbose=True)

<class 'pandas.DataFrame'>
RangeIndex: 105996 entries, 0 to 105995
Data columns (total 20 columns):
 #   Column                  Non-Null Count   Dtype         
---  ------                  --------------   -----         
 0   ID                      105996 non-null  int64         
 1   UPA                     105996 non-null  int64         
 2   EventDate               105996 non-null  datetime64[us]
 3   City                    105979 non-null  str           
 4   State                   105996 non-null  str           
 5   Primary NAICS           105994 non-null  object        
 6   Hospitalized            105996 non-null  float64       
 7   Amputation              105989 non-null  float64       
 8   Loss of Eye             105991 non-null  float64       
 9   Final Narrative         105996 non-null  str           
 10  Nature                  105996 non-null  int64         
 11  NatureTitle             105995 non-null  str           
 12  Part of Body            105996 non-null  

In [23]:
# Колонки Hospitalized, Amputation, Loss of Eye, Secondary Source -> целочисленный тип.
# Обращаемся по именам, а не по номерам: номера "съезжают", если ячейка с drop
# ещё не выполнялась (тогда под индексом 7 оказывается State со значениями вроде 'NEW YORK').
# Тип Int64 (nullable), т.к. в этих колонках есть пропуски.
int_cols = ['Hospitalized', 'Amputation', 'Loss of Eye', 'Secondary Source']
df[int_cols] = df[int_cols].astype('Int64')
df.info(verbose=True)

<class 'pandas.DataFrame'>
RangeIndex: 105996 entries, 0 to 105995
Data columns (total 20 columns):
 #   Column                  Non-Null Count   Dtype         
---  ------                  --------------   -----         
 0   ID                      105996 non-null  int64         
 1   UPA                     105996 non-null  int64         
 2   EventDate               105996 non-null  datetime64[us]
 3   City                    105979 non-null  str           
 4   State                   105996 non-null  str           
 5   Primary NAICS           105994 non-null  object        
 6   Hospitalized            105996 non-null  Int64         
 7   Amputation              105989 non-null  Int64         
 8   Loss of Eye             105991 non-null  Int64         
 9   Final Narrative         105996 non-null  str           
 10  Nature                  105996 non-null  int64         
 11  NatureTitle             105995 non-null  str           
 12  Part of Body            105996 non-null  

In [24]:
# Проверка пригодности датафрейма для реляционной БД

n = len(df)
nulls = df.isna().sum()
report = pd.DataFrame({
    'dtype': df.dtypes.astype(str),
    'nulls': nulls,
    'pct': (nulls / n * 100).round(2),
})

print(f'Строк: {n}, колонок: {df.shape[1]}')
print(f'Колонок без пропусков: {(nulls == 0).sum()} из {df.shape[1]}')
print(f'Строк без единого пропуска: {df.notna().all(axis=1).sum()}')
print()
if (nulls > 0).any():
    print('Колонки с пропусками:')
    print(report[report.nulls > 0].to_string())
else:
    print('Пропусков нет нигде.')

# Кандидат на первичный ключ
print()
print(f'ID уникален: {df.ID.is_unique} (дубликатов: {df.ID.duplicated().sum()})')
print(f'Полных дублей строк: {df.duplicated().sum()}')

Строк: 105996, колонок: 20
Колонок без пропусков: 11 из 20
Строк без единого пропуска: 50605

Колонки с пропусками:
                         dtype  nulls    pct
City                       str     17   0.02
Primary NAICS           object      2   0.00
Amputation               Int64      7   0.01
Loss of Eye              Int64      5   0.00
NatureTitle                str      1   0.00
EventTitle                 str      1   0.00
SourceTitle                str      1   0.00
Secondary Source         Int64  55381  52.25
Secondary Source Title     str  55382  52.25

ID уникален: False (дубликатов: 5)
Полных дублей строк: 0


In [25]:
# Приведение датафрейма к виду без NULL для загрузки в реляционную БД.
# Ячейку можно выполнять повторно: каждый шаг проверяет, не сделан ли он уже.

# 1. Первичный ключ: исходный ID не уникален (счётчик внутри месяца иногда совпадает
#    у разных происшествий), а UPA — номер дела в системе OSHA — уникален.
#    Поэтому удаляем ID и переименовываем UPA в ID. Строки при этом не теряются.
if 'UPA' in df.columns:
    df = df.drop(columns='ID').rename(columns={'UPA': 'ID'})
assert df['ID'].is_unique, 'ID (бывший UPA) не уникален'

# 2. Числовые колонки: пропуски -> 0, тип -> int64
int_cols = ['Hospitalized', 'Amputation', 'Loss of Eye', 'Secondary Source']
df[int_cols] = df[int_cols].fillna(0).astype('int64')

# 3. Primary NAICS хранится как object (смесь чисел и строк) -> приводим к строке
df['Primary NAICS'] = df['Primary NAICS'].fillna('unknown').astype('str')

# 4. Остальные текстовые колонки: пропуски -> 'unknown'
str_cols = df.select_dtypes(include=['object', 'str']).columns
df[str_cols] = df[str_cols].fillna('unknown').astype('str')

print(f'Строк: {len(df)}, осталось пропусков: {df.isna().sum().sum()}')
df.info(verbose=True)

Строк: 105996, осталось пропусков: 0
<class 'pandas.DataFrame'>
RangeIndex: 105996 entries, 0 to 105995
Data columns (total 19 columns):
 #   Column                  Non-Null Count   Dtype         
---  ------                  --------------   -----         
 0   ID                      105996 non-null  int64         
 1   EventDate               105996 non-null  datetime64[us]
 2   City                    105996 non-null  str           
 3   State                   105996 non-null  str           
 4   Primary NAICS           105996 non-null  str           
 5   Hospitalized            105996 non-null  int64         
 6   Amputation              105996 non-null  int64         
 7   Loss of Eye             105996 non-null  int64         
 8   Final Narrative         105996 non-null  str           
 9   Nature                  105996 non-null  int64         
 10  NatureTitle             105996 non-null  str           
 11  Part of Body            105996 non-null  int64         
 12  Part

In [26]:
df['Primary NAICS'].unique()


<StringArray>
['922140', '339999', '484121', '424490', '326122', '444110', '331492',
 '213112', '213111', '722211',
 ...
 '221116', '221117', '611513', '315222', '423520', '111160', '533110',
 '212291', '532283', '315110']
Length: 1288, dtype: str

In [27]:
# первая строка обычно служебная, реальные заголовки — во второй
naics_titles = pd.read_csv(
    "../Raw/2022_NAICS_Structure.csv",
    skiprows=2  # если верхние строки — заголовок/пустая строка, проверьте после первого прогона
)

naics_titles.columns = naics_titles.columns.str.strip()

In [28]:
naics_titles=naics_titles.drop(columns=['Unnamed: 3', 'Unnamed: 4','Unnamed: 5'])

In [29]:
naics_titles=naics_titles.drop(columns=['Change Indicator'])

In [30]:
naics_titles.tail()

,2022 NAICS Code,2022 NAICS Title
2139,9281,National Security and International Affairs
2140,92811,National Security
2141,928110,National Security
2142,92812,International Affairs
2143,928120,International Affairs


In [33]:
# ============================================================
#  Экспорт df (OSHA) и naics_titles в 6 CSV под схему БД:
#  nature_lookup, body_part_lookup, event_lookup, source_lookup,
#  naics_lookup и факт-таблица injuries.
#  Рассчитано на df после ячейки очистки выше (ID = бывший UPA, NULL заполнены).
# ============================================================
from pathlib import Path

import pandas as pd

OUT_DIR = Path("DataBaseFiles")
OUT_DIR.mkdir(parents=True, exist_ok=True)

assert 'UPA' not in df.columns and df['ID'].is_unique, \
    'Сначала выполните ячейку очистки df (ID = бывший UPA)'


def clean_title(s: pd.Series) -> pd.Series:
    """Пробелы по краям и двойные пробелы убираем; 'unknown' (заглушка из очистки) -> пропуск."""
    s = s.astype('str').str.strip().str.replace(r'\s+', ' ', regex=True)
    return s.mask(s.isin(['', 'unknown', 'nan']))


def build_lookup(*pairs: pd.DataFrame) -> pd.DataFrame:
    """
    Справочник (code, title) из одной или нескольких пар колонок «код + название».
    В данных OSHA у одного кода встречаются разные написания (разные редакции OIICS,
    лишние пробелы), а code должен быть PK — поэтому берём самый частый вариант.
    """
    lk = pd.concat([p.set_axis(['code', 'title'], axis=1) for p in pairs], ignore_index=True)
    lk = lk.dropna(subset=['code'])
    lk['code'] = lk['code'].astype('int64')
    lk['title'] = clean_title(lk['title'])

    best = (
        lk.dropna(subset=['title'])
        .groupby(['code', 'title']).size().reset_index(name='n')
        .sort_values(['code', 'n'], ascending=[True, False])
        .drop_duplicates('code')[['code', 'title']]
    )
    codes = pd.DataFrame({'code': lk['code'].drop_duplicates().sort_values()})
    out = codes.merge(best, on='code', how='left')
    out['title'] = out['title'].fillna('Unknown')  # title NOT NULL
    return out


# ---------- Вторичный источник: 0 (заглушка из очистки) -> NULL ----------
secondary = df['Secondary Source'].astype('Int64')
secondary = secondary.mask(secondary == 0)
has_secondary = secondary.notna()

# ---------- 1–4. Справочники из самих данных ----------
nature_lookup = build_lookup(df[['Nature', 'NatureTitle']])
body_part_lookup = build_lookup(df[['Part of Body', 'Part of Body Title']])
event_lookup = build_lookup(df[['Event', 'EventTitle']])
source_lookup = build_lookup(
    df[['Source', 'SourceTitle']],
    pd.DataFrame({'code': secondary[has_secondary],
                  'title': df.loc[has_secondary, 'Secondary Source Title']}),
)

# ---------- 5. naics_lookup из 2022 NAICS Structure ----------
naics_lookup = (
    naics_titles[['2022 NAICS Code', '2022 NAICS Title']]
    .set_axis(['code', 'title'], axis=1)
    .dropna(subset=['code'])
)
naics_lookup['code'] = naics_lookup['code'].astype('str').str.strip()
# census.gov помечает буквой T в конце названия коды, общие для США/Канады/Мексики
naics_lookup['title'] = (
    naics_lookup['title'].astype('str').str.strip()
    .str.removesuffix('T').str.strip()
)
naics_lookup = naics_lookup.drop_duplicates('code')

naics_code = df['Primary NAICS'].astype('str').str.strip()
naics_code = naics_code.mask(naics_code == 'unknown')  # нет кода -> NULL

# Коды из старых редакций NAICS (2012/2017), которых нет в 2022, добавляем
# в справочник с заглушкой — иначе FK injuries.naics_code не пройдёт
legacy = sorted(set(naics_code.dropna()) - set(naics_lookup['code']))
naics_lookup = pd.concat(
    [naics_lookup, pd.DataFrame({'code': legacy, 'title': 'Not in 2022 NAICS (legacy code)'})],
    ignore_index=True,
)

# ---------- 6. injuries ----------
severity_code = (
    (df['Hospitalized'] > 0).astype('int64') * 4    # Hospitalized — это число людей, а не 0/1
    + (df['Amputation'] > 0).astype('int64') * 2
    + (df['Loss of Eye'] > 0).astype('int64') * 1
)

injuries = pd.DataFrame({
    'id': df['ID'].astype('int64'),
    'event_date': pd.to_datetime(df['EventDate']).dt.strftime('%Y-%m-%d'),
    'city': df['City'],
    'state': df['State'],
    'naics_code': naics_code,
    'final_narrative': df['Final Narrative'],
    'severity_code': severity_code,
    'nature_code': df['Nature'].astype('int64'),
    'part_of_body_code': df['Part of Body'].astype('int64'),
    'event_code': df['Event'].astype('int64'),
    'source_code': df['Source'].astype('int64'),
    'secondary_source_code': secondary,
})

# ---------- Проверки PK / FK / CHECK перед записью ----------
lookups = {
    'naics_lookup': naics_lookup,
    'nature_lookup': nature_lookup,
    'body_part_lookup': body_part_lookup,
    'event_lookup': event_lookup,
    'source_lookup': source_lookup,
}
for name, lk in lookups.items():
    assert lk['code'].is_unique, f'{name}: дубли code'
    assert lk['title'].notna().all(), f'{name}: пустой title'

assert injuries['id'].is_unique
assert injuries['severity_code'].between(0, 7).all()

fks = {
    'naics_code': naics_lookup,
    'nature_code': nature_lookup,
    'part_of_body_code': body_part_lookup,
    'event_code': event_lookup,
    'source_code': source_lookup,
    'secondary_source_code': source_lookup,
}
for col, lk in fks.items():
    orphans = injuries[col].dropna()
    orphans = orphans[~orphans.isin(lk['code'])]
    assert orphans.empty, f'{col}: {orphans.nunique()} кодов без записи в справочнике'

# ---------- Запись (порядок = порядок загрузки в PostgreSQL) ----------
for name, frame in [*lookups.items(), ('injuries', injuries)]:
    frame.to_csv(OUT_DIR / f'{name}.csv', index=False)
    print(f'{name + ".csv":24} {len(frame):>7} строк')

print(f'\nCSV сохранены в {OUT_DIR}')
print(f'NAICS-кодов не из редакции 2022 (добавлены заглушкой): {len(legacy)}')
print(f'Случаев без вторичного источника (NULL): {(~has_secondary).sum()}')

naics_lookup.csv            2442 строк
nature_lookup.csv            251 строк
body_part_lookup.csv         205 строк
event_lookup.csv             476 строк
source_lookup.csv           1643 строк
injuries.csv              105996 строк

CSV сохранены в DataBaseFiles
NAICS-кодов не из редакции 2022 (добавлены заглушкой): 317
Случаев без вторичного источника (NULL): 55381
